# This notebook was used to look the saved grad cam values for Transformer-40k. The saved grad cam values were generated from the CNN_interpretability notebook.

Wanted to classify some of the peaks that changed (increased or decreased) through the layers. Used MEME suite to see if there were any motifs present in these cases

In [ ]:
import numpy as np
import sys
import time
import h5py
from tqdm import tqdm
import pyfastx
import numpy as np
import re
from math import ceil
from sklearn.metrics import average_precision_score
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import pickle
#import pickle5 as pickle
import os
from sklearn.model_selection import train_test_split

from scipy.sparse import load_npz
from glob import glob

from transformers import get_constant_schedule_with_warmup
from sklearn.metrics import precision_score,recall_score,accuracy_score

from src.train import trainModel
from src.dataloader import getData,spliceDataset,h5pyDataset,collate_fn, getDataPointListFull
from src.weight_init import keras_init
from src.losses import categorical_crossentropy_2d
from src.model import SpliceFormer, SpliceAI_10K
from src.evaluation_metrics import print_topl_statistics
import copy
from collections import defaultdict

In [8]:
data_dir = '../Data'
SL=5000
CL_max=40000
setType = 'test'
annotation_test, transcriptToLabel_test, seqData = getData(data_dir, setType)
BATCH_SIZE = 1
specific_anno = annotation_test[annotation_test['name'].apply(lambda x: x.split('--')[0])=='CFTR'] # Again looking at CFTR gene

In [16]:
path = '../Results/Grad-CAM/CFTR_Transformer45k_skips_4.pkl'

In [9]:
CL_max = 40000
SL = 5000
device = torch.device('cpu')
test_dataset = spliceDataset(getDataPointListFull(specific_anno,transcriptToLabel_test,SL,CL_max,shift=SL))
test_dataset.seqData = seqData
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
for i,(batch_chunks,target_chunks) in enumerate(tqdm(test_loader)):
    batch_features = batch_chunks.to(torch.float32).to(device)      # shape: [1, 4, 45000]
    targets = torch.squeeze(target_chunks.to(torch.float32).to(device), dim=0)  # shape: [3, 45000]
    
    signal = batch_features[0].sum(dim=0)  # shape: [45000]
    nonzero_positions = torch.nonzero(signal > 0).squeeze()
    print("Transcript spans:", nonzero_positions.min().item(), "to", nonzero_positions.max().item())

    if i == 4:
        break

  0%|          | 0/38 [00:00<?, ?it/s]/Users/douglasflorizone/Summer 2025 Research/Code/Spliceformer/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 11%|█         | 4/38 [00:00<00:00, 47.09it/s]

Transcript spans: 20000 to 44999
Transcript spans: 15000 to 44999
Transcript spans: 10000 to 44999
Transcript spans: 5000 to 44999
Transcript spans: 0 to 44999


In [10]:
# Function to remove one hot encoding
def one_hot_decode(batch_features):
    one_hot = batch_features.squeeze(0)
    idx_to_base = ['A', 'C', 'G', 'T']
    base_indices = torch.argmax(one_hot, dim=0)
    sequence = ''.join(idx_to_base[i] for i in base_indices.tolist())
    return sequence

In [11]:
def default_inner_dict():
    return defaultdict(list)

In [12]:
# Function for extracting the nucleotides around the index of interest
def extract_regions(index_dict, nucleotides, window=100):
    all_indices = []
    for category, indices in index_dict.items():
        for idx in indices:
            all_indices.append((idx, category))
    # Sort by position
    all_indices.sort(key=lambda x: x[0])
    regions = {cat: [] for cat in index_dict}
    
    previous_end = -1

    # This is to make sure that the windows between two indices don't overlap. If they do remove the one that is overlapping
    for idx, category in all_indices:
        start = max(0, idx - window)
        end = min(len(nucleotides), idx + window)
        if start < previous_end:
            continue
        
        seq = nucleotides[start:end]
        regions[category].append({
            'index': idx,
            'sequence': seq,
            'start': start,
            'end': end
            })
        previous_end = end
    
    return regions

In [13]:
# Creating fasta files to be used with MEME
def create_fasta(relevant_regions,key):
    for category,regions in relevant_regions.items():
        fasta_dir = '../Results/Grad-CAM/MEME/'
        os.makedirs(fasta_dir,exist_ok=True)
        if regions != []:
            fasta_file = os.path.join(fasta_dir,f'CFTR_Transformer_skip_4_{key}_{category}.fasta')
            with open(fasta_file,'w') as f:
                for i,region in enumerate(regions,start=1):
                    idx = region["index"]
                    seq = region["sequence"]
                    start = region["start"]
                    end = region["end"]
                    f.write(f'>{category}_{i}_idx{idx}_start{start}_end{end}\n')
                    for j in range(0,len(seq),60): #only 60 characters per line to standardize
                        f.write(f'{seq[j:j+60]}' + '\n')

In [71]:
def seq_finder(batch_features,targets,path):
    with open(path, 'rb') as f:
        grad_cam_dict = pickle.load(f)
    acceptor_idxs = np.where(targets[1,:]==1)
    donor_idxs = np.where(targets[2,:]==1)
    splice_idxs = np.where(targets[1:,:]==1)
    start,end = 0,45000
    start_shift = 20000
    shift = 20000
    nucleotides = one_hot_decode(batch_features)
    for label,key in zip(splice_idxs[0],splice_idxs[1]):
        grad_cam_full = []
        for i,layer in enumerate(grad_cam_dict[key]):
            grad_cam_key = torch.stack(grad_cam_dict[key][layer])
            grad_cam_key = torch.mean(grad_cam_key,dim=0)
            relu = torch.nn.ReLU()
            grad_cam_key = relu(grad_cam_key)
            grad_cam_key = grad_cam_key.detach().numpy()
            if grad_cam_key.max()  != np.float32(0):
                grad_cam_key /= grad_cam_key.max()
            grad_cam_full.append(grad_cam_key)
        differences = grad_cam_full[0] - grad_cam_full[3]
        # Current conditions for positive, negative and neutral cases.
        positives = np.where(differences>0.6)[0]
        negatives = np.where(differences<-0.3)[0]
        neutrals = np.where((grad_cam_full[0] > 0.4) & ( (differences > -0.3) & (differences < 0.3) ))[0]
        print(key,negatives)
        # tester = np.where((grad_cam_full[0] > 0.6) & ( (differences > -0.2) & (differences < 0.2) ))[0]
        # print(tester,key)
        index_dict = {'positives': positives, 'negatives': negatives, 'neutrals': neutrals}
        relevant_regions = extract_regions(index_dict, nucleotides)
        create_fasta(relevant_regions,key)


In [72]:
seq_finder(batch_features,targets,path)


24290 [24272 24273 24274 24275 24276 24278 24282 24290]
29071 [24290]
184 [24400 29179]
24400 [    4     5     6 ... 44246 44512 44518]
29179 [  171 24253 24258 24260 24261 24262 24263 24264 24265 24266 24268 24269
 24270 24271 24272 24273 24274 24275 24276 24277 24279 24280 24283 24387
 24395 24397 24399 24405 24413]


In [40]:
i = 0
for seq in pyfastx.Fasta('../Results/Grad-CAM/MEME/CFTR_Transformer_skip_4_29071_positives.fasta'):
    print(f"Sequence: {seq}")
    i += 1
print(i)

Sequence: GCTTGAGCCCAGACGGCCCTAGCAGGGACCCCAGCGCCCGAGAGACCATGCAGAGGTCGCCTCTGGAAAAGGCCAGCGTTGTCTCCAAACTTTTTTTCAGGTGAGAAGGTGGCCAACCGAGCTTCGGAAAGACACGTGCCCACGAAAGAGGAGGGCGTGTGTATGGGTTGGGTTTGGGGTAAAGGAATAAGCAGTTTTTA
Sequence: AAAAATTGTGTTTCACATGGCCTTACCAGATATACAGGAAACACGTCACATGTTTCTATTGTATGTTGTTAAATGCCTTAGAATTTAACTTTCTGAATAGGATCCCTTCAGTTTGAGAGTCATAAAAGAGTAAAATTATTATGGTATGAGTTATAGATTGTATTGAATATCTCTTTATATGTCTAGGTTTTGTCATTGGA
Sequence: AAGACAATTCTTTTGTTTGTTTGTTTTTAAAAGACAGAGTCTCACTCTGTTGCCCAGGCTAGAGTGCAGTGACACAATCATAACTCACTGCAACCTCCACCTCCTGGGCTCAAGTGAGCCTTCCATCTTGCCTCACGAGTAGCTGGGTCTTCAGGTGTACAGGTGTGTACCACCATGCCTGGCTAACTTTTTTTTTTTTT
Sequence: GTCTGAAGTTTGTCTGACATACTAAGCAATGTAATTAAAGTAGAAGTCGCCTAAGCTCAGCACTTTATTATGCCTTGAAATTATACTGCCTGTCCTACAGGTGAAGGTGTTATGAATGCAGTTTGTCACTGTAACTCTATTCATAGCTCTGAAAGGCTGAGAGTGACTCAGAAGAATATTTTTGCTCTGAATATGAAGAA
Sequence: TAGAGAACACTAGGTATTGGGGCTCATAGTGTGAAAACCACTGACTTAATTCTTCCCCCATCTTGGTTGTTCCTGATCTTCCCTTGTGTCCCCATTCCAGCCATTTGTATCCTTAGAAAATGATCTCATATTCTACTTCATCTTTA

I installed meme locally using: https://meme-suite.org/meme/doc/download.html and tried to run it with my fasta file from the command line. While this worked, it gave slightly different results compared to the web version of MEME Suite. So I ended up just using the web version for the few tests I did.

In [45]:
import subprocess
from pathlib import Path
fasta_file = '../Results/Grad-CAM/MEME/CFTR_Transformer_skip_4_29071_positives.fasta'
output_path= Path('../Results/Grad-CAM/MEME/MEME_outputs/')
os.makedirs(output_path,exist_ok=True)

meme_exe = str(Path.home() / "meme/bin/meme")

# MEME command
cmd = [
    meme_exe, fasta_file,
    "-oc", str(output_path),   # output directory
    "-dna",                   # DNA mode
    "-mod", "zoops",          # motif site distribution
    "-nmotifs", "5",          # number of motifs
    "-minw", "6",              # min motif width
    "-maxw", "50",              # max motif width
    "-seed", "0"
]

# Run MEME
subprocess.run(cmd, check=True)
print("MEME run complete.")

The output directory '../Results/Grad-CAM/MEME/MEME_outputs' already exists.
Its contents will be overwritten.
BACKGROUND: using background model of order 0
PRIMARY (classic): n 35 p0 35 p1 0 p2 0
SEQUENCE GROUP USAGE-- Starts/EM: p0; Trim: p0; pvalue: p0; nsites: p0,p1,p2
SEEDS: maxwords 7000 highwater mark: seq 35 pos 194
Initializing the motif probability tables for 2 to 35 sites...
nsites = 35
Done initializing.

seqs=    35, min_w= 200, max_w=  200, total_size=     7000

motif=1
SEED DEPTHS: 2 4 8 16 32 35
SEED WIDTHS: 6 8 11 15 21 29 41 50
em: w=  50, psites=  35, iter=  20 Warning: Cannot convert EPS file to PNG as no install of Image Magick or Ghostscript is usable.

motif=2
SEED DEPTHS: 2 4 8 16 32 35
SEED WIDTHS: 6 8 11 15 21 29 41 50
em: w=  50, psites=  35, iter=  10 Warning: Cannot convert EPS file to PNG as no install of Image Magick or Ghostscript is usable.

motif=3
SEED DEPTHS: 2 4 8 16 32 35
SEED WIDTHS: 6 8 11 15 21 29 41 50
em: w=  50, psites=  35, iter=  20 Warning

MEME run complete.
